# Model Training and Evaluation

## Objective

This notebook compares machine-learning models for binary cardiovascular disease classification using the UCI Heart Disease dataset.

## Research Questions

1. How does Logistic Regression compare with tree-based models?
2. Do different models show consistent performance under stratified cross-validation?
3. What trade-offs exist between precision, recall, F1-score and ROC-AUC?

## Experimental Strategy

- Use the existing training and test split.
- Keep preprocessing inside each model pipeline to prevent data leakage.
- Compare models using the same stratified cross-validation folds.
- Select models using training data only.
- Reserve the held-out test set for final evaluation after model selection.

## Scope and Limitations

The dataset contains 303 observations. Results will be interpreted cautiously because performance estimates from a small dataset can be uncertain. This project is research-oriented and is not intended for clinical diagnosis or medical decision-making.


In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split

# Loading data
df = pd.read_csv("../data/raw/heart_disease_data.csv")

# Separating predictors and target
X = df.drop(columns=["num"]).copy()
y = (df["num"] > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts().sort_index())

print("\nTest target distribution:")
print(y_test.value_counts().sort_index())

Training set: (242, 13)
Test set: (61, 13)

Training target distribution:
num
0    131
1    111
Name: count, dtype: int64

Test target distribution:
num
0    33
1    28
Name: count, dtype: int64


In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numerical_features = [
    "age", "trestbps", "chol", "thalach", "oldpeak"
]

categorical_features = [
    "sex",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "ca",
    "thal"
]

# Numerical preprocessing
numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combining both pipelines
preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

# Logistic Regression pipeline
logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Defining 5-fold stratified cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Evaluation metrics
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

# Evaluating on training data only
logistic_results = cross_validate(
    logistic_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

# results
for metric in scoring:
    scores = logistic_results[f"test_{metric}"]
    print(
        f"{metric.upper():10s}: "
        f"{scores.mean():.3f} +/- {scores.std():.3f}"
    )

ACCURACY  : 0.847 +/- 0.010
PRECISION : 0.877 +/- 0.054
RECALL    : 0.783 +/- 0.046
F1        : 0.825 +/- 0.007
ROC_AUC   : 0.902 +/- 0.014


In [17]:
print("Dataset shape:", df.shape)

print("\nTraining data shape:", X_train.shape)

print("\nTarget distribution:")
print(y_train.value_counts())

print("\nNumerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

print("\nCross-validation:")
print(cv)

Dataset shape: (303, 14)

Training data shape: (242, 13)

Target distribution:
num
0    131
1    111
Name: count, dtype: int64

Numerical features:
['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

Categorical features:
['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

Cross-validation:
StratifiedKFold(n_splits=5, random_state=42, shuffle=True)


In [18]:
# Feature definitions
num_features = [
    "age", "trestbps", "chol", "thalach", "oldpeak"
]

cat_features = [
    "sex", "cp", "fbs", "restecg",
    "exang", "slope", "ca", "thal"
]

# Fresh numerical preprocessing
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Fresh categorical preprocessing
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Fresh combined preprocessing
fresh_preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

# Fresh Logistic Regression model
fresh_model = Pipeline([
    ("preprocessor", fresh_preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Fresh cross-validation
fresh_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Evaluation
fresh_results = cross_validate(
    fresh_model,
    X_train,
    y_train,
    cv=fresh_cv,
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "roc_auc": "roc_auc"
    },
    error_score="raise"
)

# Printing individual fold scores
for metric in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    scores = fresh_results[f"test_{metric}"]

    print(f"\n{metric.upper()}")
    print("Individual folds:", scores)
    print(f"Mean: {scores.mean():.4f}")
    print(f"Std:  {scores.std():.4f}")


ACCURACY
Individual folds: [0.83673469 0.85714286 0.85416667 0.83333333 0.85416667]
Mean: 0.8471
Std:  0.0100

PRECISION
Individual folds: [0.82608696 0.94117647 0.94117647 0.81818182 0.85714286]
Mean: 0.8768
Std:  0.0542

RECALL
Individual folds: [0.82608696 0.72727273 0.72727273 0.81818182 0.81818182]
Mean: 0.7834
Std:  0.0459

F1
Individual folds: [0.82608696 0.82051282 0.82051282 0.81818182 0.8372093 ]
Mean: 0.8245
Std:  0.0069

ROC_AUC
Individual folds: [0.9180602  0.90909091 0.88986014 0.91433566 0.88111888]
Mean: 0.9025
Std:  0.0144


In [21]:
# applying Random Forest

from sklearn.ensemble import RandomForestClassifier

# Random Forest pipeline
rf_model = Pipeline(steps=[
    ("preprocessor", fresh_preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=42
    ))
])

# 5-fold cross-validation
rf_results = cross_validate(
    rf_model,
    X_train,
    y_train,
    cv=fresh_cv,
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "roc_auc": "roc_auc"
    }
)

# Displaying results
for metric in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    scores = rf_results[f"test_{metric}"]

    print(
        f"{metric.upper():10s}: "
        f"{scores.mean():.3f} +/- {scores.std():.3f}"
    )

ACCURACY  : 0.814 +/- 0.023
PRECISION : 0.797 +/- 0.027
RECALL    : 0.801 +/- 0.080
F1        : 0.796 +/- 0.036
ROC_AUC   : 0.894 +/- 0.033


### Initial Model Comparison: Logistic Regression vs Random Forest

Both models were evaluated using 5-fold stratified cross-validation on the same training dataset, with identical preprocessing and evaluation metrics.

Logistic Regression achieved better accuracy (0.847 ± 0.010), precision (0.877 ± 0.054), F1-score (0.825 ± 0.007), and ROC-AUC (0.902 ± 0.014) compared with Random Forest.

Random Forest achieved marginally higher recall (0.801 ± 0.080) than Logistic Regression (0.783 ± 0.046).

Logistic Regression also showed lower variability across all five evaluation metrics.

**Preliminary observation:** Logistic Regression currently provides the more consistent overall performance. However, further evaluation is required before selecting the final model.

### Fold-wise ROC-AUC Comparison

To examine model consistency, ROC-AUC scores were compared across the same five cross-validation folds.

This comparison helps identify whether one model performs more consistently across different training-validation splits. The results should be interpreted cautiously because the dataset is relatively small.

In [22]:
roc_auc_comparison = pd.DataFrame({
    "Fold": [f"Fold {i}" for i in range(1, 6)],
    "Logistic Regression": fresh_results["test_roc_auc"],
    "Random Forest": rf_results["test_roc_auc"]
})

roc_auc_comparison["Difference (LR - RF)"] = (
    roc_auc_comparison["Logistic Regression"]
    - roc_auc_comparison["Random Forest"]
)

display(roc_auc_comparison)

,Fold,Logistic Regression,Random Forest,Difference (LR - RF)
0,Fold 1,0.918060,0.949833,-0.031773
1,Fold 2,0.909091,0.849327,0.059764
2,Fold 3,0.889860,0.884615,0.005245
3,Fold 4,0.914336,0.904720,0.009615
4,Fold 5,0.881119,0.879371,0.001748


### Fold-wise ROC-AUC Analysis

The fold-wise comparison showed that Logistic Regression achieved a higher ROC-AUC in four out of five folds, while Random Forest performed better in one fold.

Logistic Regression demonstrated a higher mean ROC-AUC (0.902 ± 0.014) compared with Random Forest (0.894 ± 0.033).

The lower standard deviation suggests that Logistic Regression had more consistent performance across the five folds. However, the difference in average ROC-AUC was relatively small, and these results do not establish statistically significant superiority.

**Preliminary observation:** Logistic Regression is currently the stronger candidate based on overall cross-validation performance.

In [23]:
# performing hyperparameter tuning

from sklearn.model_selection import GridSearchCV

param_grid = {
    "classifier__C": [0.001, 0.01, 0.1, 1, 10, 100]
}

grid_search = GridSearchCV(
    estimator=fresh_model,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=fresh_cv,
    n_jobs=-1,
    return_train_score=True
)

grid_search.fit(X_train, y_train)

print("Best C:", grid_search.best_params_["classifier__C"])
print("Best Cross-Validation ROC-AUC:", grid_search.best_score_)

Best C: 1
Best Cross-Validation ROC-AUC: 0.9024931590148981


In [24]:
# displaying results

tuning_results = pd.DataFrame(grid_search.cv_results_)

tuning_summary = tuning_results[
    ["param_classifier__C", "mean_test_score", "std_test_score"]
].copy()

tuning_summary = tuning_summary.rename(columns={
    "param_classifier__C": "C",
    "mean_test_score": "Mean ROC-AUC",
    "std_test_score": "ROC-AUC Std"
})

display(tuning_summary.sort_values("C"))

,C,Mean ROC-AUC,ROC-AUC Std
0,0.001,0.875229,0.035265
1,0.010,0.889338,0.034652
2,0.100,0.902456,0.016560
3,1.000,0.902493,0.014440
4,10.000,0.899468,0.013674
5,100.000,0.897733,0.012093


### Logistic Regression Hyperparameter Tuning

GridSearchCV was used to evaluate six regularisation strengths (`C`) using five-fold stratified cross-validation.

The best-performing value was C = 1, achieving a mean ROC-AUC of 0.90249 ± 0.01444.

The value C = 0.1 produced a nearly identical mean ROC-AUC of 0.90246 ± 0.01656.

Very strong regularisation (C = 0.001 and C = 0.01) resulted in lower ROC-AUC scores. Increasing C beyond 1 did not improve performance.

**Conclusion:** Hyperparameter tuning did not produce a meaningful improvement over the original Logistic Regression baseline. The default C = 1 remains a reasonable choice.

In [25]:
best_model = grid_search.best_estimator_

print(best_model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'trestbps', 'chol',
                                                   'thalach', 'oldpeak']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_un

In [26]:
# evaluating on untouched dataset

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Generating predictions
y_pred = best_model.predict(X_test)

# predicted probabilities
y_prob = best_model.predict_proba(X_test)[:, 1]

# Calculating evaluation metrics
print("FINAL TEST SET PERFORMANCE")

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

FINAL TEST SET PERFORMANCE
Accuracy : 0.8852459016393442
Precision: 0.8387096774193549
Recall   : 0.9285714285714286
F1 Score : 0.8813559322033898
ROC-AUC  : 0.9664502164502166

Confusion Matrix:
[[28  5]
 [ 2 26]]

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.85      0.89        33
           1       0.84      0.93      0.88        28

    accuracy                           0.89        61
   macro avg       0.89      0.89      0.89        61
weighted avg       0.89      0.89      0.89        61



### Final Evaluation on the Held-out Test Set

After hyperparameter tuning, the selected Logistic Regression pipeline (C = 1) was evaluated on the untouched test set containing 61 observations.

The model achieved:

- Accuracy: 88.52%
- Precision: 83.87%
- Recall: 92.86%
- F1-score: 88.14%
- ROC-AUC: 96.65%

The confusion matrix showed 28 true negatives, 26 true positives, 5 false positives and 2 false negatives.

The final test performance was higher than the five-fold cross-validation averages. This difference may reflect variation in the small test sample and should not be interpreted as proof of superior generalisation.

**Overall finding:** Logistic Regression provides a promising baseline for classifying recorded cardiovascular disease status in this dataset. Further interpretation and validation are required before drawing broader conclusions.